## 버즈매트릭스 TV 스마트 기능 회귀분석 (v2 with 가중 상관분석)

---

### 1. 분석 목적
TV 스마트 기능 카테고리 간 **감성점수(SC Score)의 상호 영향 관계**를 파악하여, 특정 기능의 VOC 감성이 다른 기능 감성에 미치는 영향도를 정량화한다.

---

### 2. 데이터 정의

| 구분 | 테이블 | 설명 |
| --- | --- | --- |
| 모델 메타 | `sandbox.z_jungryo_lee.buzz_model_meta` | model_id, country, year, brand_name, model, d_type |
| VOC 점수 | `sandbox.z_jungryo_lee.buzz_sc_score_dataset` | model_id, category, total_count, avg_sc |

**분석 단위**: model_id (제품 모델)  
**종속변수 (Y)**: 각 카테고리의 `avg_sc` (가중평균 감성점수)  
**독립변수 (X)**: Y를 제외한 나머지 카테고리의 `avg_sc`  
**가중치**: `sqrt(total_count)` — VOC 건수가 많은 모델에 더 큰 가중치 부여

---

### 3. 분석 대상 피처 (15개 스마트 카테고리)

| 코드 | 카테고리명 |
| --- | --- |
| 07_01 | 채널/컨텐츠 APP |
| 07_02 (1) | 구동성/구동속도 - TV 전반 |
| 07_02 (2) | 구동성/구동속도 - APP/웹 전반 |
| 07_03 | 메뉴 구성/UI |
| 07_04 | SW/OS |
| 07_05 | 컨텐츠 탐색 사용성 |
| 07_06 | 리모컨 사용성 |
| 07_07 | 리모컨 디자인 |
| 07_08 | 음성 인식/조작 |
| 07_09 | 게임 기능 |
| 07_10 | 부가 기능 |
| 07_11 | 홈 IoT |
| 07_12 | 모바일 연동 |
| 07_13 | 광고 |
| 07_20 | 전반적 스마트 사용성 |

---

### 4. 분석 방법론

1. **가중 상관분석 (Weighted Correlation)**: X 후보군 중 |가중상관| ≥ 0.1인 변수만 선별
2. **가중최소제곱 회귀 (WLS)**: 선별된 X로 모델 적합, 가중치 = `sqrt(total_count)`
3. **그룹별 반복 수행**: 전체(all) / 브랜드 / 국가 / 연도 / 디바이스 타입 기준으로 분리 분석

---

### 5. 진행 현황 및 결과 요약

| 항목 | 값 |
| --- | --- |
| 회귀모델 수 | 239건 |
| 계수 상세 (전체) | 2,664건 |
| 유의한 계수 (p < 0.05) | 497건 (18.7%) |

**결과 저장 테이블**:
- `sandbox.z_jungryo_lee.wls_model_summary_with_corr` — 모델 요약 (variable1, y_feature, R², n_obs)
- `sandbox.z_jungryo_lee.wls_coef_detail_with_corr` — 계수 상세 (x_feature, coef, p_value, weighted_corr)

---

### 6. 코드 구조

| Cell | 내용 |
| --- | --- |
| Cell 1 | 데이터 로드 및 Wide 변환 (score/count 피벗) |
| Cell 2 | 컬럼명 클렌징 함수 |
| Cell 3 | 가중 상관계수 함수 정의 |
| Cell 4 | 가중 상관 기반 X 변수 선택 함수 |
| Cell 5 | WLS 회귀 실행 함수 |
| Cell 6 | 전체/그룹별 반복 회귀 수행 (메인 루프) |
| Cell 7 | 결과 Delta 테이블 저장 |
| Cell 8 | 유의한 계수 필터링 (p < 0.05) |
| Cell 9 | R² 분포 시각화 |

In [0]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import statsmodels.api as sm

spark = SparkSession.builder.getOrCreate()

# 1) 테이블 로드
df_meta = spark.table("sandbox.z_jungryo_lee.buzz_model_meta").toPandas()
df_voc = spark.table("sandbox.z_jungryo_lee.buzz_sc_score_dataset").toPandas()

# 2) JOIN
df_long = df_voc.merge(df_meta, on="model_id", how="left")

# 3) Wide 변환
df_score = (
    df_long
    .pivot_table(index="model_id", columns="category", values="avg_sc")
    .add_suffix("_score")
)

df_count = (
    df_long
    .pivot_table(index="model_id", columns="category", values="total_count")
    .add_suffix("_count")
)

df_wide = pd.concat([df_score, df_count], axis=1).fillna(0)

In [0]:
def clean_name(s: str) -> str:
    return (
        str(s)
        .replace('.', '')
        .replace('/', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '_')
        .replace(' ', '_')
    )

def clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

def clean_feature_pool(feature_pool: list) -> list:
    return [clean_name(f) for f in feature_pool]

df_wide = clean_col_names(df_wide)

In [0]:
def weighted_corr(x, y, w):
    x = np.asarray(x)
    y = np.asarray(y)
    w = np.asarray(w)

    w_sum = np.sum(w)
    if w_sum == 0:
        return np.nan

    mx = np.sum(w * x) / w_sum
    my = np.sum(w * y) / w_sum

    cov = np.sum(w * (x - mx) * (y - my))
    var_x = np.sum(w * (x - mx) ** 2)
    var_y = np.sum(w * (y - my) ** 2)

    if var_x == 0 or var_y == 0:
        return np.nan

    return cov / np.sqrt(var_x * var_y)

In [0]:
def select_x_by_weighted_corr(
    pdf,
    y_feature,
    x_features,
    corr_threshold=0.1
):
    y_score_col = f"{y_feature}_score"
    y_count_col = f"{y_feature}_count"

    y = pdf[y_score_col]
    w = np.sqrt(pdf[y_count_col])  # ✅ 권장 1

    selected = []

    for x in x_features:
        if x == y_feature:  # ✅ Y 변수는 X 후보에서 제외
            continue
        x_score_col = f"{x}_score"
        if x_score_col not in pdf.columns:
            continue

        corr = weighted_corr(pdf[x_score_col], y, w)

        if pd.notna(corr) and abs(corr) >= corr_threshold:
            selected.append((x, corr))

    return selected

In [0]:
def run_wls_weighted(pdf, y_feature, x_features, corr_threshold=0.1):

    y_score_col = f"{y_feature}_score"
    y_count_col = f"{y_feature}_count"

    pdf = pdf[
        (pdf[y_count_col] > 0) &
        pdf[y_score_col].notna()
    ]

    if pdf.empty:
        return None, None

    # 1) 가중 상관으로 X 필터
    selected_x = select_x_by_weighted_corr(
        pdf, y_feature, x_features, corr_threshold
    )

    if len(selected_x) == 0:
        return None, None

    final_x = [x for x, _ in selected_x]

    # 2) WLS
    X = pdf[[f"{x}_score" for x in final_x]]
    X = sm.add_constant(X)
    y = pdf[y_score_col]

    weights = np.sqrt(pdf[y_count_col])  # ✅ 권장 1과 일관

    model = sm.WLS(y, X, weights=weights).fit()

    return model, selected_x

In [0]:
FEATURE_POOL = [
    "07_01 채널_컨텐츠 APP", "07_02 구동성_구동속도_(1)TV 전반",
    "07_02 구동성_구동속도_(2)APP_웹전반", "07_03 메뉴 구성_UI",
    "07_04 SW_OS", "07_05 컨텐츠 탐색 사용성",
    "07_06 리모컨 사용성", "07_07 리모컨 디자인",
    "07_08 음성 인식_조작", "07_09 게임 기능",
    "07_10 부가 기능", "07_11 홈 IoT",
    "07_12 모바일 연동", "07_13 광고",
    "07_20 전반적 스마트 사용성"
]

FEATURE_POOL_CLEAN = clean_feature_pool(FEATURE_POOL)

model_rows = []
coef_rows = []

for variable1 in ["all", "brand_name", "country", "year", "d_type"]:

    if variable1 == "all":
        group_keys = ["all"]
    else:
        group_keys = df_meta[variable1].dropna().unique()

    for key in group_keys:

        if variable1 == "all":
            pdf = df_wide.copy()
        else:
            model_ids = df_meta[df_meta[variable1] == key]["model_id"].unique()
            pdf = df_wide.loc[df_wide.index.isin(model_ids)]

        if pdf.empty:
            continue

        for y_feature in FEATURE_POOL_CLEAN:
            model, corr_info = run_wls_weighted(
                pdf, y_feature, FEATURE_POOL_CLEAN, corr_threshold=0.0
            )

            if model is None:
                continue

            model_rows.append({
                "y_feature": y_feature,
                "y_obs": int(model.nobs),
                "r_squared": float(model.rsquared),
                "adj_r_squared": float(model.rsquared_adj),
                "f_statistic": float(model.fvalue) if pd.notna(model.fvalue) else None,
                "prob_f": float(model.f_pvalue) if pd.notna(model.f_pvalue) else None,
                "log_likelihood": float(model.llf),
                "aic": float(model.aic),
                "bic": float(model.bic),
                "cond_no": float(model.condition_number),
                "group_dim": variable1,
                "group_key": str(key)
            })

        # β₀ (intercept) 저장
            coef_rows.append({
                "variable1": variable1,
                "variable1_value": key,
                "y_feature": y_feature,
                "x_feature": "β₀",
                "coef": float(model.params.get("const")) if pd.notna(model.params.get("const")) else None,
                "p_value": float(model.pvalues.get("const")) if pd.notna(model.pvalues.get("const")) else None,
                "t_value": float(model.tvalues.get("const")) if pd.notna(model.tvalues.get("const")) else None,
                "x_obs": int(model.nobs),
                "weighted_corr": None
            })

            for x, wcorr in corr_info:
                x_param = f"{x}_score"

                coef_rows.append({
                    "variable1": variable1,
                    "variable1_value": key,
                    "y_feature": y_feature,
                    "x_feature": x,
                    "coef": float(model.params.get(x_param)) if pd.notna(model.params.get(x_param)) else None,
                    "p_value": float(model.pvalues.get(x_param)) if pd.notna(model.pvalues.get(x_param)) else None,
                    "t_value": float(model.tvalues.get(x_param)) if pd.notna(model.tvalues.get(x_param)) else None,
                    "x_obs": int(model.nobs),
                    "weighted_corr": float(wcorr) if pd.notna(wcorr) else None
                })

In [0]:
import pandas as pd

# 1) 결과 저장
df_model = pd.DataFrame(model_rows)
df_coef = pd.DataFrame(coef_rows)

spark.createDataFrame(df_model).write.mode("overwrite").saveAsTable(
    "sandbox.z_jungryo_lee.wls_model_summary_with_corr"
)
spark.createDataFrame(df_coef).write.mode("overwrite").saveAsTable(
    "sandbox.z_jungryo_lee.wls_coef_summary_with_corr"
)

print(f"모델 요약: {len(df_model)}건 저장 완료")
print(f"계수 상세: {len(df_coef)}건 저장 완료")


In [0]:
df_sig = df_coef[df_coef["p_value"] < 0.05].sort_values(
    ["variable1", "y_feature", "p_value"]
).reset_index(drop=True)

print(f"유의한 계수: {len(df_sig)}건 / 전체 {len(df_coef)}건")
display(spark.createDataFrame(df_coef))

In [0]:
# 3) R² 분포 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 R² 히스토그램
axes[0].hist(df_model["r2"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("R² Distribution (All Groups)", fontsize=13)
axes[0].set_xlabel("R²")
axes[0].set_ylabel("Count")
axes[0].axvline(df_model["r2"].median(), color="red", linestyle="--", label=f'Median={df_model["r2"].median():.3f}')
axes[0].legend()

# 그룹별 R² 박스플롯
groups = df_model.groupby("variable1")["r2"].apply(list)
axes[1].boxplot(groups.values, labels=groups.index)
axes[1].set_title("R² by Grouping Variable", fontsize=13)
axes[1].set_ylabel("R²")
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()